In [ ]:
!pip install --upgrade transformers
!pip install -q -U bitsandbytes
!pip install -q -U accelerate
!pip install peft
!pip install optuna
!pip install evaluate
!pip install mauve-text
!pip install bert_score
!pip install hf_xet
!pip install git+https://github.com/google-research/bleurt.git

## load & import library

In [ ]:
import time
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import prepare_model_for_kbit_training, LoraConfig, PeftModel, get_peft_model
from datasets import load_dataset, Dataset
from accelerate import Accelerator
import pandas as pd
import os
import re
import pickle
import optuna
import copy
import json

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import kagglehub

# Download Deepseek latest version
base_model_path = kagglehub.model_download("deepseek-ai/deepseek-r1/transformers/deepseek-r1-distill-qwen-7b")

In [ ]:
# Muat model dan tokenizer DeepSeek
# base_model_path = "/kaggle/input/deepseek-r1/transformers/deepseek-r1-distill-qwen-7b/2"

accelerator = Accelerator()
device = accelerator.device
print(device)

# Konfigurasi quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Muat model dan tokenizer DeepSeek
base_model = AutoModelForCausalLM.from_pretrained(base_model_path, quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(
    base_model_path,
    padding_side="left",
    device_map="auto",
    add_eos_token=True
)
tokenizer.pad_token = tokenizer.eos_token

## load dataset

In [ ]:
# Muat dataset

train_data = pd.read_parquet("writing-prompts/wp_train.parquet")
test_data = pd.read_parquet("writing-prompts/wp_test.parquet")
val_data = pd.read_parquet("writing-prompts/wp_valid.parquet")

train_dataset = Dataset.from_pandas(train_data.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_data.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_data.reset_index(drop=True))

In [ ]:
# Tokenisasi dataset
def tokenize(prompt):
    result = tokenizer(
        prompt,
        truncation=True,
        max_length=512,
        padding="max_length",
    )
    result["labels"] = result["input_ids"].copy()
    return result

def generate_and_tokenize_prompt(data_point):
    full_prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are a creative writer specializing in short story writing. Generate a story based on the given details with minimum 1000 words.
    
### Input:
Story prompt (topic/idea): {data_point["prompt"]}
Story theme: {data_point["theme"]}

### Response:
{data_point["story"]}
"""
    return tokenize(full_prompt)

tokenized_train_dataset = train_dataset.map(generate_and_tokenize_prompt)
tokenized_val_dataset = val_dataset.map(generate_and_tokenize_prompt)
tokenized_test_dataset = test_dataset.map(generate_and_tokenize_prompt)

In [ ]:
# print(tokenized_val_dataset[0])

### sampling

In [ ]:
def balanced_sample(df, total_samples=52):
    theme_counts = df['theme'].value_counts()
    num_themes = len(theme_counts)
    samples_per_theme = total_samples // num_themes
    remainder = total_samples % num_themes

    sorted_themes = theme_counts.sort_values(ascending=False).index.tolist()

    sample_df = pd.DataFrame()
    for i, theme in enumerate(sorted_themes):
        n_samples = samples_per_theme + (1 if i < remainder else 0)
        theme_subset = df[df['theme'] == theme]

        if len(theme_subset) >= n_samples:
            sample = theme_subset.sample(n=n_samples, random_state=42)
        else:
            sample = theme_subset

        sample_df = pd.concat([sample_df, sample], ignore_index=True)

    return sample_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Apply balanced sampling to both val and test
sampled_val_df = balanced_sample(val_data, total_samples=52)
sampled_test_df = balanced_sample(test_data, total_samples=52)

# Convert to Hugging Face Datasets
sampled_val_dataset = Dataset.from_pandas(sampled_val_df)
sampled_test_dataset = Dataset.from_pandas(sampled_test_df)

# Tokenize
tokenized_sampled_val_dataset = sampled_val_dataset.map(generate_and_tokenize_prompt)
tokenized_sampled_test_dataset = sampled_test_dataset.map(generate_and_tokenize_prompt)

In [ ]:
# print(tokenized_sampled_val_dataset[0])

## metrics

In [ ]:
import evaluate
import torch
import numpy as np
import re
from tqdm import tqdm
import math

def generate_texts_from_model(model, dataset, max_length=2048):
    """Generate texts while preserving alignment with reference stories"""
    model.eval()
    results = []

    small_dataset = dataset.select(range(2))  
    
    for sample in tqdm(small_dataset, desc="Generating stories"):
        prompt_text = f"""You are a creative writer specializing in short story writing. Generate a story based on the given details with minimum 1000 words. Only output the story!

        ### Input:
        Story prompt (topic/idea): {sample["prompt"]}
        Story theme: {sample["theme"]}

        ### Response:
        """
        
        inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=max_length).to("cuda")
        
        with torch.no_grad():
            output = model.generate(
                input_ids=inputs["input_ids"],
                max_new_tokens=max_length,
                do_sample=True,
                top_p=0.9,
                temperature=1.5,
                repetition_penalty=1.2,
                pad_token_id=tokenizer.eos_token_id,
                attention_mask=inputs["attention_mask"],
            )
        
        generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
        cleaned_text = clean_generated_text(generated_text)
        
        results.append({
            "generated_text": cleaned_text,
            "reference_text": sample["story"],
            "prompt": sample["prompt"],
            "theme": sample["theme"]
        })

    return results

def clean_generated_text(text):
    """Clean generated text by removing prompt artifacts"""
    # Remove everything before the actual story
    cleaned_text = re.sub(r".*?</think>", "", text, flags=re.DOTALL).strip()
    return cleaned_text.strip()

def evaluate_perplexity(model, generated_results, eval_loss):
    print("Evaluating perplexity")

    if eval_loss == 0:
        total_loss = 0
        num_samples = 0
        
        for result in generated_results:
            inputs = tokenizer(result["generated_text"], return_tensors="pt", truncation=True, max_length=512)
            inputs = {k: v.to("cuda") for k, v in inputs.items()}
            labels = inputs["input_ids"]
    
            with torch.no_grad():
                outputs = model(**inputs, labels=labels)
                loss = outputs.loss.item()
    
            total_loss += loss
            num_samples += 1

        return np.exp(total_loss / num_samples)
    else:
        return math.exp(eval_loss)

def evaluate_bleu(generated_results):
    print("Evaluating BLEU")
    bleu = evaluate.load("bleu")
    predictions = [res["generated_text"] for res in generated_results]
    references = [[res["reference_text"]] for res in generated_results]
    results = bleu.compute(predictions=predictions, references=references)
    return results["bleu"]

def evaluate_bertscore(generated_results):
    print("Evaluating BERTScore")
    bertscore = evaluate.load("bertscore")
    predictions = [res["generated_text"] for res in generated_results]
    references = [res["reference_text"] for res in generated_results]
    results = bertscore.compute(predictions=predictions, references=references, lang="en")
    return {
        "precision": np.mean(results["precision"]),
        "recall": np.mean(results["recall"]),
        "f1": np.mean(results["f1"])
    }

def evaluate_mauve(generated_results):
    print("Evaluating MAUVE")
    mauve = evaluate.load("mauve")
    predictions = [res["generated_text"] for res in generated_results]
    references = [res["reference_text"] for res in generated_results]
    results = mauve.compute(predictions=predictions, references=references)
    return results.mauve

def evaluate_distinct_n(generated_results, n=2):
    """Calculate Distinct-N metric (diversity of n-grams)"""
    print(f"Evaluating Distinct-{n}")
    generated_texts = [res["generated_text"] for res in generated_results]
    
    all_ngrams = set()
    total_ngrams = 0
    
    for text in generated_texts:
        tokens = text.split()
        ngrams = set(zip(*[tokens[i:] for i in range(n)]))
        all_ngrams.update(ngrams)
        total_ngrams += len(ngrams)
    
    return len(all_ngrams) / total_ngrams if total_ngrams > 0 else 0

def evaluate_repetition_n(generated_results, n=2):
    """Calculate Repetition-N metric (fraction of repeated n-grams)"""
    print(f"Evaluating Repetition-{n}")
    generated_texts = [res["generated_text"] for res in generated_results]
    
    repeated_ngrams = 0
    total_ngrams = 0
    
    for text in generated_texts:
        tokens = text.split()
        ngrams = list(zip(*[tokens[i:] for i in range(n)]))
        total_ngrams += len(ngrams)
        repeated = len(ngrams) - len(set(ngrams))
        repeated_ngrams += max(0, repeated)

    return repeated_ngrams / total_ngrams if total_ngrams > 0 else 0
    
def evaluate_bleurt(generated_results):
    print("Evaluating BLEURT")
    bleurt = evaluate.load("bleurt", module_type="metric")
    predictions = [res["generated_text"] for res in generated_results]
    references = [res["reference_text"] for res in generated_results]
    results = bleurt.compute(predictions=predictions, references=references)
    return np.mean(results["scores"])

def run_full_evaluation(model, dataset, eval_loss=0):
    """Complete evaluation pipeline with separated metrics"""
    # 1. Generate aligned texts
    generation_results = generate_texts_from_model(model, dataset)
    
    # 2. Run all evaluations
    metrics = {
        "perplexity": evaluate_perplexity(model, generation_results, eval_loss),
        "bleu": evaluate_bleu(generation_results),
        "bleurt": evaluate_bleurt(generation_results),
        "bertscore": evaluate_bertscore(generation_results),
        "mauve": evaluate_mauve(generation_results),
        "distinct_4": evaluate_distinct_n(generation_results, n=4),
        "repetition_4": evaluate_repetition_n(generation_results, n=4),
    }
    
    # 3. Add metadata
    metrics.update({
        "num_samples": len(generation_results),
        "avg_length": np.mean([len(res["generated_text"].split()) for res in generation_results])
    })
    
    return metrics, generation_results

## study

In [ ]:
def get_num_layers(model):
    numbers = set()
    for name, _ in model.named_parameters():
        for number in re.findall(r'\d+', name):
            numbers.add(int(number))
    return max(numbers)

def get_last_layer_linears(model):
    names = []
    
    num_layers = get_num_layers(model)
    for name, module in model.named_modules():
        if str(num_layers) in name and not "encoder" in name:
            if isinstance(module, torch.nn.Linear):
                names.append(name)
    return names

In [ ]:
def objective(trial):    
    global start_time
    if time.time() - start_time > TIME_LIMIT:
        raise optuna.exceptions.TrialPruned()

    torch.cuda.empty_cache()

    # Hyperparameter search space
    # lora_r = trial.suggest_categorical('r', [4, 8, 16, 32])
    lora_r = 4

    # Mapping alpha dari rank
    alpha_mapping = {
        4: [8, 16],
        8: [16, 32],
        16: [32, 64],
        32: [64, 128]
    }
    lora_alpha = trial.suggest_categorical('alpha', alpha_mapping[lora_r])

    dropout = trial.suggest_categorical('dropout', [0.0, 0.1, 0.01])
    # learning_rate = trial.suggest_categorical('learning_rate', [1e-6, 1e-5, 1e-4])
    learning_rate = 1e-4
    batch_size = 1
    # grad_accum_steps = trial.suggest_categorical('grad_accum_steps', [1, 2, 4, 8, 16, 32])
    grad_accum_steps = 16
    num_epochs = 1

    # Validasi agar batch size total tidak melebihi ukuran dataset
    effective_batch_size = batch_size * grad_accum_steps
    if effective_batch_size > len(tokenized_train_dataset):
        raise optuna.exceptions.TrialPruned()

    print(f"Trial {trial.number} sedang berjalan dengan parameter:")
    print(f"  r={lora_r}, alpha={lora_alpha}, dropout={dropout}, learning_rate={learning_rate}, batch_size={batch_size}, grad_accum_steps={grad_accum_steps}, num_epochs={num_epochs}")

    # Setup LoRA configuration
    config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        target_modules=get_last_layer_linears(base_model),
        bias="none",
        lora_dropout=dropout,
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(base_model, config)

    training_args = TrainingArguments(
        output_dir=f"./optuna_trial_{trial.number}",
        warmup_ratio=0.1,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum_steps,
        num_train_epochs=num_epochs,
        learning_rate=learning_rate,
        logging_steps=50,
        fp16=True,
        optim="paged_adamw_8bit",
        logging_dir="./logs",
        report_to='none',
    )

    trainer = Trainer(
        model=model,
        train_dataset=tokenized_train_dataset,
        eval_dataset=tokenized_val_dataset,
        args=training_args,
        data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )

    model.config.use_cache = False
    trainer.train()
    eval_result = trainer.evaluate()

    try:
        # Try full evaluation
        metrics, results = run_full_evaluation(model, tokenized_sampled_val_dataset, eval_result["eval_loss"])

        # Store all metrics
        trial.set_user_attr("eval_loss", float(eval_result["eval_loss"]))
        trial.set_user_attr("bertscore_f1", float(metrics["bertscore"]["f1"]))
        for k, v in metrics.items():
            if k not in ["bertscore"]:
                trial.set_user_attr(k, v)

        # Objective score
        score = (metrics["mauve"] + metrics["distinct_4"] + metrics["bleu"]) - (eval_result["eval_loss"] + metrics["repetition_4"])

        # Print results
        print(f"Metrics: {metrics}")

        # Sample output
        print("\n=== Sample Generation ===")
        for i, res in enumerate(results[:2]):
            print(f"\nSample {i+1} (Prompt: {res['prompt'][:50]}...)")
            print(f"Generated: {res['generated_text'][:100]}...")
            print(f"Reference: {res['reference_text'][:100]}...")

        # Save output
        os.makedirs("trial_outputs", exist_ok=True)
        save_generated_text(metrics, results, f"trial_outputs/trial_{trial.number}.json")
    except Exception as e:
        print(f"\n⚠️ Evaluation failed: {e}")
        score = -eval_result["eval_loss"]
        trial.set_user_attr("eval_loss", float(eval_result["eval_loss"]))

    del model
    torch.cuda.empty_cache()

    return metrics["perplexity"], metrics["mauve"], metrics["bleurt"]

In [ ]:
def save_generated_text(metrics, results, filename='trial_outputs/generated.json'):
    output_data = dict(metrics)
    output_data["samples"] = results
    
    # Write results to JSON
    with open(filename, mode='w', encoding='utf-8') as f:
        json.dump(output_data, f, indent=2, ensure_ascii=False)

    print("generated text saved successfully.")

In [ ]:
def save_study(study, filename='last.pkl'):
    with open(filename, 'wb') as f:
        pickle.dump(study, f)
    print("Study saved successfully.")

def save_callback(study, trial):
    save_study(study)

In [ ]:
def select_best_trial(study, metric="perplexity"):
    """
    Pilih trial terbaik dari Optuna multi-objective study berdasarkan salah satu metrik.
    Args:
        study (optuna.Study): Study Optuna multi-objektif
        metric (str): Salah satu dari 'perplexity', 'mauve', 'bleurt'
    
    Returns:
        best_trial (optuna.trial.FrozenTrial): Trial terbaik untuk metrik yang dipilih
    """
    if metric not in ["perplexity", "mauve", "bleurt"]:
        raise ValueError("Metric harus salah satu dari 'perplexity', 'mauve', atau 'bleurt'.")

    # Pemetaan metrik ke indeks dalam directions
    metric_index = {
        "perplexity": 0,
        "mauve": 1,
        "bleurt": 2
    }[metric]

    # Ambil semua best trials (pareto optimal)
    best_trials = study.best_trials

    # Urutkan sesuai arah optimasi
    if study.directions[metric_index] == optuna.study.StudyDirection.MINIMIZE:
        best_trial = sorted(best_trials, key=lambda t: t.values[metric_index])[0]
    else:
        best_trial = sorted(best_trials, key=lambda t: t.values[metric_index], reverse=True)[0]

    return best_trial

def print_trials(path):
    with open(path, 'rb') as f:
        study = pickle.load(f)

    # Tampilkan informasi umum tentang studi
    print("Study name:", study.study_name)
    print("Number of trials:", len(study.trials))
    
    # Tampilkan trial terbaik
    print("Best trial:")
    best_perplexity_trial = select_best_trial(study, "perplexity")
    print("Best (Perplexity):", best_perplexity_trial.values, best_perplexity_trial.params)
    
    best_mauve_trial = select_best_trial(study, "mauve")
    print("Best (MAUVE):", best_mauve_trial.values, best_mauve_trial.params)
    
    best_bleurt_trial = select_best_trial(study, "bleurt")
    print("Best (BLEURT):", best_bleurt_trial.values, best_bleurt_trial.params)
    
    # Tampilkan semua trials
    for trial in study.trials:
        print(f"Trial {trial.number}:")
        print(f"  Params: {trial.params}")
        print(f"  Value: {trial.values}")
        print(f"  Attributes: {trial.user_attrs}")

In [ ]:
base_model = base_model.to('cuda').train()

TIME_LIMIT = 11 * 3600
start_time = time.time()

study_path = 'last.pkl'

# Muat hasil optuna sebelumnya jika ada
try:
    with open(study_path, 'rb') as f:
        study = pickle.load(f)
    print("Loaded existing Optuna study.")
    print_trials(study_path)
except FileNotFoundError:
    study = optuna.create_study(study_name="deepseek-r1-qwen7b-wp", directions=["minimize", "maximize", "maximize"])

try:
    study.optimize(objective, timeout=TIME_LIMIT, callbacks=[save_callback], n_trials=1)
except (KeyboardInterrupt, optuna.exceptions.TrialPruned, Exception) as e:
    print(f"Training berhenti karena: {e}")
    save_study(study)

print_trials('last.pkl')